# Camada Gold: Eficiência Emocional e Regras de Negócio

Este notebook finaliza o pipeline aplicando regras de negócio para responder à pergunta: **Qual o impacto no desempenho dos times diante de um gol sofrido?**

Utilizaremos as tabelas da Camada Silver para calcular métricas avançadas com PySpark (`Window Functions`).

In [ ]:
# --- CONFIGURAÇÃO AGNÓSTICA DO AMBIENTE ---
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("FootballPipeline_Gold").getOrCreate()

ENVIRONMENT = os.getenv("SPARK_ENV", "databricks")

if ENVIRONMENT == "databricks":
    PATH_SILVER = "/Volumes/workspace/silver/football_processed"
    PATH_GOLD = "/Volumes/workspace/gold/football_analytics"
    CATALOG_NAME = "workspace"
else:
    PATH_SILVER = "./data/silver"
    PATH_GOLD = "./data/gold"
    CATALOG_NAME = "local"

print(f"Ambiente Configurado: {ENVIRONMENT}")

In [ ]:
# 1. Leitura da Camada Silver
if ENVIRONMENT == "databricks":
    df_events = spark.table(f"{CATALOG_NAME}.silver.goal_events")
else:
    df_events = spark.read.format("delta").load(f"{PATH_SILVER}/goal_events")

print(f"Total de eventos de gols na Silver: {df_events.count()}")
df_events.show(5)

In [ ]:
# 2. Lógica Cronológica: Encontrando o 'Próximo Gol' de cada partida
# Usamos Window Functions para ordenar os gols pelo minuto dentro de cada partida

window_match = Window.partitionBy("match_id").orderBy("minute")

# Criamos colunas virtuais que 'olham para a próxima linha' (próximo gol da mesma partida)
df_timeline = df_events.withColumn("next_scoring_team", F.lead("scoring_team").over(window_match)) \
                       .withColumn("next_minute", F.lead("minute").over(window_match))

# Calcula a diferença de tempo (em minutos) até o próximo gol
df_timeline = df_timeline.withColumn("time_diff_to_next_goal", F.col("next_minute") - F.col("minute"))

df_timeline.filter(F.col("next_scoring_team").isNotNull()).show(5)

In [ ]:
# 3. Regras de Negócio: Reação vs Apagão

# CENÁRIO 1: O time sofreu o gol (conceding_team) e ele mesmo marcou o próximo gol (next_scoring_team == conceding_team)
# Isso significa que o time REAGIU.
df_reaction = df_timeline.filter(F.col("next_scoring_team") == F.col("conceding_team")) \
                         .withColumn("is_reaction", F.lit(1)) \
                         .withColumn("is_collapse", F.lit(0))

# CENÁRIO 2: O time sofreu o gol (conceding_team) e o adversário marcou DE NOVO (next_scoring_team != conceding_team)
# Isso significa um APAGÃO ou Efeito Dominó.
df_collapse = df_timeline.filter(F.col("next_scoring_team") != F.col("conceding_team")) \
                         .withColumn("is_reaction", F.lit(0)) \
                         .withColumn("is_collapse", F.lit(1))

# Unindo os cenários analisados
df_analyzed = df_reaction.union(df_collapse)

In [ ]:
# 3.5 Visão Granular (Atendendo ao FOUNDATION.md)
# Criando uma tabela no nível da Partida/Evento com o final_result e goals_scored_after_conceding

# Conta quantos gols o time fez APÓS sofrer o gol na mesma partida
window_after_conceding = Window.partitionBy("match_id", "conceding_team").orderBy("minute").rowsBetween(0, Window.unboundedFollowing)
df_analyzed = df_analyzed.withColumn("goals_scored_after_conceding", F.sum("is_reaction").over(window_after_conceding))

# O Resultado Final (final_result) pode ser inferido pelo saldo ou cruzado com silver.matches
# Para simplificar nesta visão, vamos apenas focar na contagem de gols de resposta:
df_gold_events = df_analyzed.select(
    "match_id",
    "conceding_team",
    F.col("minute").alias("goal_minute"),
    "goals_scored_after_conceding"
)

if ENVIRONMENT == "databricks":
    df_gold_events.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG_NAME}.gold.match_emotional_events")
else:
    df_gold_events.write.format("delta").mode("overwrite").save(f"{PATH_GOLD}/match_emotional_events")

print("✅ Tabela gold.match_emotional_events criada com sucesso (Visão Granular)!")

In [ ]:
# 4. Agregação Final (Por Equipe)
# Responde exatamente a pergunta do FOUNDATION.md

df_gold = df_analyzed.groupBy("conceding_team").agg(
    F.count("match_id").alias("total_gols_sofridos"),
    F.sum("is_reaction").alias("total_reacoes"),
    F.round(F.mean(F.when(F.col("is_reaction") == 1, F.col("time_diff_to_next_goal"))), 2).alias("media_tempo_reacao_min"),
    F.sum("is_collapse").alias("total_apagoes"),
    F.round(F.mean(F.when(F.col("is_collapse") == 1, F.col("time_diff_to_next_goal"))), 2).alias("media_tempo_apagao_min")
).withColumnRenamed("conceding_team", "equipe") \
 .orderBy(F.col("media_tempo_reacao_min").asc_nulls_last())

# As equipes no topo da tabela são as mais 'Eficientes Emocionalmente' (Reagem mais rápido)
df_gold.show(10)

In [ ]:
# 5. Salvando na Camada Gold
if ENVIRONMENT == "databricks":
    df_gold.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG_NAME}.gold.emotional_efficiency")
else:
    df_gold.write.format("delta").mode("overwrite").save(f"{PATH_GOLD}/emotional_efficiency")

print("✅ Camada Gold criada com sucesso! Base de dados pronta para Dashboard BI.")